## Train clould

In [25]:
!git clone https://github.com/nbngoc123/multilingual-nmt.git

fatal: destination path 'multilingual-nmt' already exists and is not an empty directory.


In [26]:
%cd /content/multilingual-nmt

/content/multilingual-nmt


In [ ]:
import os
os.chdir("/content/multilingual-nmt")

FileNotFoundError: [WinError 2] The system cannot find the file specified: '/multilingual-nmt'

In [28]:
!git branch -a

  main
* training
  remotes/origin/HEAD -> origin/main
  remotes/origin/develop
  remotes/origin/feature/data
  remotes/origin/feature/inference
  remotes/origin/feature/model
  remotes/origin/feature/training
  remotes/origin/main


In [29]:
!git checkout -b training origin/feature/training

fatal: A branch named 'training' already exists.


In [1]:
%cd /content/multilingual-nmt

import os
import sys

print("Current dir:", os.getcwd())

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

/content/multilingual-nmt
Current dir: /content/multilingual-nmt


### Train local

In [1]:
import os
import sys

sys.path.append(os.path.abspath('..'))
os.chdir('..')

from collections import Counter

from src.utils.logger import setup_logger
from src.utils.seed import set_seed
from src.utils.config import load_config
from src.data.loader import load_multilingual_dataset
from src.data.preprocessor import MultilingualPreprocessor
from src.data.collator import custom_collate_fn
from src.model.tokenizer import load_mbart_tokenizer
from src.model.builder import load_mbart_model
from src.training.arguments import get_training_args
from src.training.callbacks import get_callbacks
from src.training.trainer import BalancedLossSeq2SeqTrainer
from src.evaluation.metrics import get_compute_metrics

# 1. Môi trường cơ sở
logger = setup_logger("train_script")
set_seed(42)
logger.info("Bắt đầu kịch bản huấn luyện NMT Đa Ngôn Ngữ với Balanced Loss!")

# 2. Tải cấu hình
data_config = load_config("configs/data_config.yaml")
train_config = load_config("configs/train_config.yaml")
model_config = load_config("configs/model_config.yaml")

: 

In [ ]:
logger.info("Đang tải dữ liệu OPUS-100...")
dataset = load_multilingual_dataset(
    lang_pairs=data_config["lang_pairs"],
    max_samples=data_config["max_samples"],
    val_samples=data_config["val_samples"],
    test_samples=data_config["test_samples"]
)

In [ ]:
# Tính tần suất ngôn ngữ từ tập Train để chia trọng số phạt
pair_counts = dict(Counter(dataset["train"]["pair"]))
logger.info(f"Tần suất cặp ngôn ngữ (phục vụ Balanced Loss): {pair_counts}")

# 4. Tải Model & Tokenizer
tokenizer = load_mbart_tokenizer(model_config["model_name"])
model = load_mbart_model(model_config["model_name"])

# 5. Tiền xử lý dữ liệu (Tokenization)
logger.info("Tiến hành Tokenize toàn bộ tập dữ liệu...")
preprocessor = MultilingualPreprocessor(tokenizer, max_length=data_config["max_length"])

tokenized_datasets = dataset.map(
    preprocessor.preprocess_function,
    batched=True,
    batch_size=1000,
    remove_columns=["src", "tgt"] # Xóa cột văn bản thô, nhưng GIỮ LẠI cột 'pair'
)

In [ ]:
# 6. Cấu hình Training
training_args = get_training_args(train_config, output_dir="./saved_models/mbart50-balanced")
callbacks = get_callbacks()

# 7. Khởi tạo BalancedLossSeq2SeqTrainer
logger.info("Gắn kết Model, Data, Loss vào Trainer...")
trainer = BalancedLossSeq2SeqTrainer(
    pair_counts=pair_counts,
    smoothing_factor=0.5,
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=custom_collate_fn,
    compute_metrics=get_compute_metrics(tokenizer),
    callbacks=callbacks
)

# 8. Bắt đầu Vòng lặp Huấn luyện (Training Loop)
logger.info("🚀 Bắt đầu HUẤN LUYỆN!")
trainer.train()

# 9. Lưu trữ sau khi chạy xong
logger.info("Đang lưu mô hình hoàn chỉnh...")
trainer.save_model("./saved_models/mbart50-balanced-final")
tokenizer.save_pretrained("./saved_models/mbart50-balanced-final")
logger.info("Tuyệt vời! Đã hoàn tất huấn luyện!")



[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


2026-06-11 08:05:42 - train_script - INFO - Gắn kết Model, Data, Loss vào Trainer...
2026-06-11 08:05:45 - train_script - INFO - 🚀 Bắt đầu HUẤN LUYỆN!


Epoch,Training Loss,Validation Loss


In [ ]:
!pip install -q sacrebleu
!pip install evaluate

In [ ]:
!git pull origin

In [ ]:
!pip install --upgrade datasets pyarrow fsspec

In [ ]:
import json
import os

os.makedirs("outputs", exist_ok=True)
with open("outputs/training_history.json", "w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, indent=4)

print("Đã lưu lịch sử huấn luyện!")


In [ ]:
!zip -r mbart50-balanced-final.zip ./saved_models/mbart50-balanced-final

In [ ]:
from google.colab import files
files.download("mbart50-balanced-final.zip")

In [ ]:
from src.utils.config import load_config
from src.model.tokenizer import load_mbart_tokenizer
from src.model.builder import load_mbart_model
from src.inference.translator import MBartTranslator

# 1. Trỏ đường dẫn tới thư mục mô hình BẠN VỪA TRAIN XONG
model_path = "./saved_models/mbart50-balanced-final"

print("Đang tải Tokenizer và Model...")
tokenizer = load_mbart_tokenizer(model_path)
model = load_mbart_model(model_path)

# 2. Khởi tạo bộ dịch
translator = MBartTranslator(model, tokenizer)
inference_config = load_config("configs/inference_config.yaml")

# 3. Dịch thử nghiệm
text_to_translate = "Hello, how are you today? I am learning artificial intelligence."

print(f"\n[Tiếng Anh]: {text_to_translate}")

# Thử dịch sang Tiếng Việt
result_vi = translator.translate(
    text=text_to_translate,
    src_lang="en",
    tgt_lang="vi",
    inference_config=inference_config
)
print(f"[Tiếng Việt]: {result_vi}")

# Thử dịch sang Tiếng Pháp (nếu model của bạn có train cặp en-fr)
result_fr = translator.translate(
    text=text_to_translate,
    src_lang="en",
    tgt_lang="fr",
    inference_config=inference_config
)
print(f"[Tiếng Pháp]: {result_fr}")


In [ ]:
!python scripts/batch_infer.py